In [407]:
import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, confusion_matrix, classification_report
from sklearn.calibration import CalibratedClassifierCV


In [408]:
df = pd.read_csv("../dataset_pede_tratado.csv")

df = df.sort_values(["RA", "ANO"])

print("\nShape original:", df.shape)


Shape original: (3030, 26)


In [409]:
df

,RA,ANO,FASE,TURMA,GENERO,ANO_NASC,ANO_INGRESSO,INSTITUICAO_DE_ENSINO,INDE,IAN,...,MATEM,PORTUG,INGLES,DEFAS,FASE_IDEAL,ATINGIU_PV,INDICADO,IDADE,FASE_PASSADA,REPETENTE_FASE
0,RA-1,2022,7,A,Feminino,2003,2016,Escola Pública,5.783000,5.0,...,2.7,3.5,6.0,-1,Fase 8 (Universitários),0,1,19,NaN,0
1,RA-1,2023,8,8E,Feminino,2003,2016,Privada *Parcerias com Bolsa 100%,7.034148,10.0,...,NaN,NaN,NaN,0,Fase 8 (Universitários),0,0,20,7.0,0
2,RA-1,2024,8,8E,Feminino,2003,2021,Privada *Parcerias com Bolsa 100%,6.060000,10.0,...,NaN,NaN,NaN,0,Fase 8 (Universitários),0,0,21,8.0,1
3,RA-10,2022,7,A,Feminino,2004,2021,Escola Pública,5.784000,5.0,...,3.3,2.6,6.4,-1,Fase 8 (Universitários),0,0,18,NaN,0
4,RA-100,2022,4,A,Feminino,2009,2019,Rede Decisão,7.618000,10.0,...,7.0,7.8,8.1,1,Fase 3 (7º e 8º ano),0,0,13,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3025,RA-995,2024,1,1E,Feminino,2012,2023,Pública,5.110372,5.0,...,0.0,4.0,NaN,-2,Fase 3 (7° e 8° ano),0,0,12,0.0,0
3026,RA-996,2023,0,ALFA T - G2/G3,Masculino,2014,2023,Pública,8.189200,5.0,...,8.7,8.5,NaN,-1,Fase 1 (3° e 4° ano),0,0,9,NaN,0
3027,RA-997,2023,0,ALFA T - G2/G3,Masculino,2015,2023,Pública,7.568700,10.0,...,7.2,7.0,NaN,0,ALFA (1° e 2° ano),0,0,8,NaN,0
3028,RA-998,2023,0,ALFA U - G2/G3,Feminino,2012,2023,Pública,7.975200,5.0,...,8.0,7.7,NaN,-2,Fase 2 (5° e 6° ano),0,0,11,NaN,0


In [410]:
indicadores = ["IDA","IEG","IAA","IPS","IPP","IPV"]

for col in indicadores:
    df[f"DELTA_{col}"] = df.groupby("RA")[col].diff()

In [411]:
df["TEMPO_PROGRAMA"] = df["ANO"] - df["ANO_INGRESSO"]
df["DEFAS_FUTURO"] = df.groupby("RA")["DEFAS"].shift(-1)
df["RISCO_FUTURO"] = (df["DEFAS_FUTURO"] <= -2).astype(int)


df_2022 = df.loc[df["ANO"] == 2022]
df_2023 = df.loc[df["ANO"] == 2023]

features = [
    "IDA",
    "IEG",
    "IAA",
    "IPS",
    "IPP",
    "IPV",
    "IDADE",
    "FASE",
    "TEMPO_PROGRAMA",
    "REPETENTE_FASE",
    "DELTA_IDA",
    "DELTA_IEG",
    "DELTA_IAA",
    "DELTA_IPS",
    "DELTA_IPP",
    "DELTA_IPV"
]

X_train, X_test, y_train, y_test = train_test_split(
    df_2022[features], df_2022["RISCO_FUTURO"], test_size=0.20, random_state=42)


In [412]:
rf = RandomForestClassifier(
    n_estimators=600,
    max_depth=5,
    min_samples_leaf=8,
    min_samples_split=20,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42   
)

rf.fit(X_train, y_train)

print(accuracy_score(y_train, rf.predict(X_train)))
print(roc_auc_score(y_train, rf.predict(X_train)))
print(precision_score(y_train, rf.predict(X_train), average=None))

df_conf = pd.DataFrame(data={"y_predict": rf.predict(X_train),
                             "y_real": y_train})

df_conf.groupby(["y_predict","y_real"], as_index=False).size().pivot_table(index="y_predict", columns="y_real", values="size")

0.8459302325581395
0.8270994772916793
[0.96998124 0.41935484]


y_real,0,1
y_predict,,
0,517.0,16.0
1,90.0,65.0


In [413]:
print(accuracy_score(y_test, rf.predict(X_test)))
print(roc_auc_score(y_test, rf.predict(X_test)))
print(precision_score(y_test, rf.predict(X_test), average=None))

df_conff = pd.DataFrame(data={"y_predict": rf.predict(X_test),
                             "y_real": y_test})

df_conff.groupby(["y_predict","y_real"], as_index=False).size().pivot_table(index="y_predict", columns="y_real", values="size")


0.8313953488372093
0.6989908546199937
[0.92957746 0.36666667]


y_real,0,1
y_predict,,
0,132.0,10.0
1,19.0,11.0


In [414]:
df_2024 = df.loc[df["ANO"] == 2024]

x = df_2024[features]

df_2024["PROB_RISCO_FUTURO"] = rf.predict_proba(x)[:,1]

print("\nAlunos com maior risco futuro previsto:")

print(

    df_2024[["RA","ANO","PROB_RISCO_FUTURO"]]

    .sort_values("PROB_RISCO_FUTURO", ascending=False)

    .head(10)
)


joblib.dump(

    rf,
    "../modelo_risco_temporal_final.pkl"

)

print("\nModelo salvo como modelo_risco_temporal_final.pkl")


Alunos com maior risco futuro previsto:
           RA   ANO  PROB_RISCO_FUTURO
2792   RA-880  2024           0.737484
1698   RA-454  2024           0.693806
2519   RA-771  2024           0.689113
29    RA-1013  2024           0.685536
938   RA-1600  2024           0.685063
2984   RA-975  2024           0.683829
1512   RA-382  2024           0.679889
1742   RA-470  2024           0.668606
1677   RA-444  2024           0.667953
914   RA-1582  2024           0.666437

Modelo salvo como modelo_risco_temporal_final.pkl


C:\Users\T-GAMER\AppData\Local\Temp\ipykernel_20052\3210888162.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2024["PROB_RISCO_FUTURO"] = rf.predict_proba(x)[:,1]
